# Лекция 07. Хеш-таблицы

Как найти нужную запись, не просматривая весь справочник? Разберём, как хеш сужает поиск, почему разные ключи могут оказаться рядом и как словари помогают обрабатывать обычные данные.

## Разминка: проверим понимание

[Открыть квиз по материалам лекций 1–6](quiz.html) — пять коротких вопросов на **5–10 минут**. Условия и пропуски в данных, ссылки и копирование, сложность циклов, импорт и сортировка.

Сначала выберите ответ без запуска кода и объясните себе, почему он верен. В каждом вопросе один правильный вариант; после всех ответов откроется разбор.

Квиз работает без интернета: скачайте `quiz.html` и откройте его в браузере.

## Цели

После лекции вы сможете:

- проследить поиск от ключа через хеш до корзины;
- объяснить, почему коллизия не означает потерю записи;
- выполнить вставку, замену, поиск, удаление и рехеширование в учебной таблице;
- назвать условия средней `O(1)` и отличить её от худшего случая;
- выбрать ключ и значение для накопления, дедупликации, индексирования и кэширования;
- заметить, когда индекс молча теряет данные или кэш смешивает разные запросы.

## Перед началом

Нужны списки, циклы, функции, `dict`, `set`, ссылки и хешируемость из занятия 5, а также оценки сложности из занятия 3.

В оценках считаем вычисление хеша и сравнение ключей операциями `O(1)`: работаем с небольшими числами и короткими идентификаторами. Для длинных строк и больших чисел их собственную стоимость нужно учитывать отдельно.

## Сначала реальная задача

Есть небольшой справочник курсов валют. Если он хранится списком, для поиска `USD` приходится просматривать записи по очереди. Один поиск на трёх строках незаметен, но миллион поисков по большому справочнику превращается в настоящую работу.

In [ ]:
rates = [("EUR", 101.2), ("USD", 92.4), ("CNY", 12.7)]

def find_rate(records, code):
    for stored_code, rate in records:
        if stored_code == code:
            return rate
    raise KeyError(code)

assert find_rate(rates, "USD") == 92.4

## Индекс меняет цену повторных поисков

Список хранит записи по позициям. Для запроса «какой курс у USD?» удобен словарь `код → курс`: один раз просматриваем справочник, затем обращаемся по коду.

Построение индекса требует `O(N)` времени в среднем и дополнительной памяти. Готовый индекс обычно даёт `O(1)` на запрос. Для одного поиска подготовка может не окупиться; для множества запросов к тем же данным она устраняет повторные проходы. Теперь посмотрим, откуда берётся быстрый доступ по ключу.

In [ ]:
rate_by_code = {code: rate for code, rate in rates}
assert rate_by_code["USD"] == 92.4
assert "GBP" not in rate_by_code

## Модель хеш-таблицы

В учебной таблице есть массив **корзин**. Каждая корзина — отдельный список пар `[key, value]`. Будем использовать **метод цепочек**: если несколько ключей выбрали одну корзину, их пары хранятся рядом.

Поиск проходит четыре шага:

1. получить целое число `hash(key)`;
2. вычислить номер корзины: `hash(key) % capacity`;
3. просмотреть пары только в этой корзине;
4. узнать ключ по идентичности или равенству: `stored_key is key or stored_key == key`.

Хеш не является адресом объекта и не обязан быть уникальным. Он выбирает место поиска; проверка ключа подтверждает, что запись та самая.

Дальше проследим одну таблицу с двумя корзинами и ключами `1`, `3`, `5`. Это учебная модель, а не описание внутреннего устройства `dict` в CPython.

![Поиск ключа 3: выбираем корзину 1, затем различаем ключи 1 и 3.](assets/lookup-route.svg)

In [ ]:
def bucket_index(key, capacity):
    return hash(key) % capacity

for key in [1, 3, 5]:
    print(key, "→ хеш", hash(key), "→ корзина", bucket_index(key, 2))

## Контракт хеша и равенства

Если хешируемые ключи равны (`a == b`), их хеши должны совпадать. Обратное неверно: одинаковый хеш не доказывает равенство. Тем более его не доказывает одинаковый остаток от деления на число корзин.

У `1` и `3` разные хеши, но при двух корзинах одинаковый остаток: оба ключа направляются в корзину `1`. Поэтому сравнение ключей внутри неё обязательно.

Хеш ключа должен оставаться постоянным, а равенство — согласованным с хешем, пока запись хранится в таблице. Встроенный список не хешируем и ключом `dict` быть не может. Хешируемость уже знакома из занятия 5; сегодня используем это требование при размещении записей.

## Коллизии неизбежны

Разные ключи могут выбрать одну корзину — это **коллизия корзин**. Она возможна и при разных полных хешах, как у `1` и `3`. Разных ключей больше, чем корзин, поэтому рассчитывать на отсутствие столкновений нельзя.

Метод цепочек сохраняет все пары в выбранном списке. Коллизия добавляет работу при поиске, но не должна уничтожать или подменять запись.

## Создаём пустую таблицу

В квизе новый внешний список сохранял ссылки на вложенные строки. Здесь похожая ошибка может сломать всю таблицу: `[[]] * 3` создаёт три ссылки на **одну** корзину.

**До запуска:** что покажет `shared[1]`, если добавить запись только через `shared[0]`? А что покажет `independent[1]`?

Comprehension создаёт отдельный список для каждой позиции. Сравним оба случая, затем заведём пустую рабочую таблицу с двумя корзинами.

![Повторение списка даёт общую корзину; comprehension создаёт независимые корзины.](assets/independent-buckets.svg)

In [ ]:
def make_table(capacity=8):
    if capacity <= 0:
        raise ValueError("capacity must be positive")
    return [[] for _ in range(capacity)]

shared = [[]] * 3
independent = make_table(3)
shared[0].append(["test", 1])
independent[0].append(["test", 1])
print("Общая корзина:", shared)
print("Независимые корзины:", independent)
assert shared[0] is shared[1]
assert independent[1] == []

table = make_table(2)
print("Рабочая таблица:", table)

## Вставка и поиск

При вставке сначала выбираем корзину и ищем ключ в ней. Если ключ найден, меняем значение и возвращаем `False`: новой записи не появилось. Иначе добавляем пару и возвращаем `True`.

Поиск выбирает ту же корзину и использует ту же проверку `is or ==`. Если ключ отсутствует, получаем `KeyError`. Причину проверки `is` на необычном ключе разберём в дополнительном блоке.

Определим операции, затем вставим `1` и `3` в нашу пустую таблицу. Перед второй вставкой предскажите содержимое обеих корзин.

In [ ]:
def table_put(table, key, value):
    bucket = table[bucket_index(key, len(table))]
    for pair in bucket:
        if pair[0] is key or pair[0] == key:
            pair[1] = value
            return False
    bucket.append([key, value])
    return True


def table_get(table, key):
    bucket = table[bucket_index(key, len(table))]
    for stored_key, value in bucket:
        if stored_key is key or stored_key == key:
            return value
    raise KeyError(key)

assert table_put(table, 1, "one") is True
print("После вставки 1:", table)
assert table_put(table, 3, "three") is True
print("После вставки 3:", table)
assert table_get(table, 1) == "one"
assert table_get(table, 3) == "three"

### Добавим ключ и заменим значение

Сейчас корзина `0` пуста, а в корзине `1` лежат `[1, "one"]` и `[3, "three"]`. При поиске `3` сначала проверяется ключ `1`, затем `3`.

**До запуска:** сколько пар будет после вставки `5`? А после повторной вставки `3` со значением `"THREE"`? Какие `bool` вернут вызовы?

Это два последовательных действия над той же таблицей. Обновление ключа не добавляет вторую пару с этим ключом.

![В одну таблицу добавляем 5, затем обновляем значение 3; новая пара появляется только на первом шаге.](assets/put-update.svg)

In [ ]:
added = table_put(table, 5, "five")
print("Добавили 5:", added, table)
assert added is True
assert table[1] == [[1, "one"], [3, "three"], [5, "five"]]

added = table_put(table, 3, "THREE")
print("Обновили 3:", added, table)
assert added is False
assert len(table[1]) == 3
assert table_get(table, 3) == "THREE"

## Удаление

В нашей корзине три пары: ключи `1`, `3`, `5`. Для удаления ищем нужную пару и удаляем её по индексу. Остальные пары продолжают храниться в цепочке.

**До запуска:** останется ли доступен ключ `5` после удаления `3`? Что вернёт повторное удаление `3`?

Правило относится к методу цепочек. В других способах разрешения коллизий удаление может быть устроено иначе.

![Из нашей цепочки удаляется пара с ключом 3; ключи 1 и 5 сохраняются.](assets/delete-pair.svg)

In [ ]:
def table_delete(table, key):
    bucket = table[bucket_index(key, len(table))]
    for index, pair in enumerate(bucket):
        if pair[0] is key or pair[0] == key:
            del bucket[index]
            return True
    return False

removed = table_delete(table, 3)
print("Удалили 3:", removed, table)
assert removed is True
assert table_get(table, 1) == "one"
assert table_get(table, 5) == "five"

removed = table_delete(table, 3)
print("Повторное удаление:", removed, table)
assert removed is False

## Коэффициент заполнения

Обозначим число пар через `N`, число корзин через `B` (`capacity`). Коэффициент заполнения — `N / B`. При хорошем распределении он описывает среднюю длину цепочки: чем она длиннее, тем больше сравнений нужно для поиска.

В нашей таблице после удаления две пары и две корзины. Вернём ключ `3`: пар станет три, корзин останется две. `table_put` сам таблицу **не расширяет** — пока она растёт только за счёт удлинения цепочки.

В следующем опыте расширим её вручную. Настоящей растущей таблице нужен ещё и механизм, который следит за заполнением и вовремя запускает расширение. Порог зависит от реализации.

![Одни и те же восемь ключей в двух и восьми корзинах: длина цепочек зависит от распределения и заполнения.](assets/load-factor.svg)

In [ ]:
def table_size(table):
    return sum(len(bucket) for bucket in table)


def load_factor(table):
    return table_size(table) / len(table)

print("После удаления: пар", table_size(table), "корзин", len(table))
assert table_size(table) == 2
assert table_put(table, 3, "THREE") is True
print("Вернули 3:", table)
print("Пар:", table_size(table), "корзин:", len(table), "заполнение:", load_factor(table))
assert table[1] == [[1, "one"], [5, "five"], [3, "THREE"]]
assert load_factor(table) == 1.5

## Resize означает рехеширование

В той же таблице лежат `1`, `5`, `3`. Хотим увеличить число корзин с двух до восьми.

**До запуска:** будет ли работать поиск `3`, если просто дописать шесть пустых корзин? Какой номер корзины теперь вычислит `table_get`?

Хеш ключа остался тем же, но остаток зависит от делителя: `hash(3) % 2 == 1`, а `hash(3) % 8 == 3`. Запись в старой корзине `1` поиск уже не увидит. Сначала проверим ошибочный способ на копии внешнего списка; рабочую таблицу сохраним для правильного переноса.

In [ ]:
wrong = table.copy()
wrong.extend(make_table(6))
print("Корзин стало:", len(wrong))
print("Поиск 3 идёт в корзину:", bucket_index(3, len(wrong)))
try:
    table_get(wrong, 3)
except KeyError:
    print("Ключ 3 остался в старой корзине и не находится.")
else:
    raise AssertionError("Простое добавление корзин не должно сохранять этот поиск")
assert len(table) == 2

### Переносим все пары

Создаём новый массив независимых корзин и заново вставляем каждую пару. После переноса ключи и значения сохраняются, меняются только номера корзин. Порядок пар внутри старой цепочки `1, 5, 3` для поиска значения не важен.

![Пары из той же таблицы с ключами 1, 5, 3 заново размещаются в восьми корзинах.](assets/rehash.svg)

In [ ]:
def table_resize(table, new_capacity):
    new_table = make_table(new_capacity)
    for bucket in table:
        for key, value in bucket:
            table_put(new_table, key, value)
    return new_table

old_table = table
table = table_resize(table, 8)
print("До расширения:", old_table)
print("После расширения:", table)
for key, expected in [(1, "one"), (3, "THREE"), (5, "five")]:
    assert table_get(table, key) == expected
    print(key, "→ корзина", bucket_index(key, len(table)))
assert table_size(table) == 3
assert load_factor(table) == 3 / 8

## Откуда берётся сложность

В корзине поиск остаётся линейным. Если ключи распределены хорошо, а число корзин растёт вместе с числом пар, средняя длина цепочки ограничена и работа с ключом занимает в среднем `O(1)`. Если все `N` пар попали в одну корзину, потребуется до `N` сравнений.

| Операция | В среднем при хорошем распределении и ограниченном заполнении | Худший случай |
|---|---:|---:|
| поиск по ключу | `O(1)` | `O(N)` |
| вставка или замена без расширения | `O(1)` | `O(N)` |
| удаление | `O(1)` | `O(N)` |
| обход всех пар | `O(B + N)` | `O(B + N)` |
| `table_size` в нашем коде | `O(B)` | `O(B)` |

`B` — число корзин. Обход пар посещает и пустые корзины. `table_size` вызывает `len` у каждой корзины; это не чтение заранее сохранённого счётчика. Для маленького примера такой код удобен, но проверять им заполнение после каждой вставки большой таблицы было бы дорого. Практическая реализация хранит число пар отдельно.

### Сколько стоит наш resize

Он проходит старые корзины, создаёт новые и вставляет пары через `table_put`. При хорошем распределении и ограниченной загрузке новой таблицы это `O(B + B_new + N)`, где `B_new` — новая ёмкость. Когда обе ёмкости соразмерны числу пар, получаем среднее `O(N)`.

В худшем случае все ключи попадают в одну новую корзину. Повторные вставки проверят `0 + 1 + 2 + … + (N − 1)` пар: наш код требует `O(B + B_new + N²)`. Поэтому обещать ему линейное худшее время нельзя.

### Среднее и амортизированное — разные объяснения

Среднее время зависит от распределения ключей. Амортизация объясняет стоимость расширений на длинной серии операций. Если при хорошем распределении расширять ёмкость вдвое, суммарная работа линейных переносов растёт как `8 + 16 + 32 + …`: она пропорциональна достигнутому размеру. Стоимость этих расширений в среднем распределяется по вставкам.

Наша учебная функция автоматически этого не делает. Условие «контролируем заполнение и расширяем геометрически» нужно добавить к алгоритму, прежде чем говорить о средней амортизированной `O(1)` вставке в растущую таблицу. Плохое распределение ключей такое расширение само по себе не исправляет.

## Накопление по ключу

Есть журнал расходов. Хотим узнать по каждой категории и число покупок, и общую сумму. В обоих случаях ключ — категория. Различается то, что хранится рядом с ней: количество или накопленная сумма.

Для новой категории начинаем с нуля. Для повторной берём прежнее значение и прибавляем `1` либо сумму очередной покупки. `mapping.get(key, 0)` возвращает сохранённое значение, а если ключа ещё нет — ноль; сам этот вызов словарь не изменяет.

**До запуска:** после второй покупки в категории «еда» сколько ключей будет в каждом словаре? Какие значения изменятся?

In [ ]:
transactions = [
    {"category": "еда", "amount": 900},
    {"category": "такси", "amount": 450},
    {"category": "еда", "amount": 300},
]

count_by_category = {}
total_by_category = {}
for transaction in transactions:
    category = transaction["category"]
    count_by_category[category] = count_by_category.get(category, 0) + 1
    total_by_category[category] = (
        total_by_category.get(category, 0) + transaction["amount"]
    )
    print(category, "→ число:", count_by_category, "сумма:", total_by_category)

assert count_by_category == {"еда": 2, "такси": 1}
assert total_by_category == {"еда": 1200, "такси": 450}

После обработки каждой покупки словари выше описывают уже прочитанную часть журнала. При `n` покупках и `k` категориях получаем ожидаемое время `O(n)` и память `O(k)`. Два словаря не превращают один проход в квадратный.

## Уникальные значения в нужном порядке

Теперь нужен список клиентов без повторов, в порядке первого появления. Здесь значение рядом с ключом не требуется: достаточно знать, встречался ли клиент. Для этого берём `seen`, а последовательность результата собираем в отдельном списке `unique`.

**Выберите подход до запуска:** хватит ли одного `set(client_ids)`? Что изменится, если проверять `client_id not in unique`?

![Множество отмечает встреченные ключи, список сохраняет порядок первых появлений.](assets/ordered-dedup.svg)

In [ ]:
client_ids = ["c-2", "c-1", "c-2", "c-3", "c-1"]
seen = set()
unique = []
for client_id in client_ids:
    if client_id not in seen:
        seen.add(client_id)
        unique.append(client_id)

print("Первые появления:", unique)
assert unique == ["c-2", "c-1", "c-3"]

Множество само по себе не сохраняет нужную очередь. Проверка в списке результата очередь сохранит, но при большом числе разных клиентов каждый следующий поиск станет длиннее: до `O(n²)` за весь проход. Сочетание списка и множества даёт ожидаемое `O(n)` времени и `O(k)` памяти, где `k` — число разных клиентов. Если требуется знать **сколько раз** клиент встречается среди рассматриваемых записей, одного `seen` уже мало: нужен счётчик.

## Индекс вместо повторного поиска

Следующая задача: присоединить баланс к каждому профилю. Порядок профилей должен сохраниться, исходные записи менять нельзя. Сначала предположим, что идентификаторы балансов уникальны и для каждого профиля баланс есть.

**До запуска:** балансы идут в обратном порядке. Сколько сравнений идентификаторов сделает такой поиск для двух профилей?

In [ ]:
profiles = [
    {"client_id": "c-1", "name": "Анна"},
    {"client_id": "c-2", "name": "Борис"},
]
balances = [
    {"client_id": "c-2", "balance": 300},
    {"client_id": "c-1", "balance": 120},
]

comparisons = 0
report_by_scan = []
for profile in profiles:
    for row in balances:
        comparisons += 1
        if row["client_id"] == profile["client_id"]:
            enriched = profile.copy()
            enriched["balance"] = row["balance"]
            report_by_scan.append(enriched)
            break

print("Сравнений:", comparisons)
print(report_by_scan)
assert comparisons == 3

Для Анны просмотрели две записи, для Бориса — одну. Если профилей `n`, а балансов `m`, такой поиск требует до `n · m` сравнений.

Построим справочник `client_id → balance` один раз. Затем каждый профиль обращается к нему по своему идентификатору. Ключ связывает записи, поэтому порядок строк в таблице балансов теперь не важен.

Заранее договоримся об ошибках: повторный идентификатор в балансах и отсутствующий баланс означают `ValueError`. Дубликат обнаруживаем **до** замены значения, иначе одна из исходных строк тихо потеряется. Нулевой баланс допустим: проверяем наличие ключа, а не истинность суммы.

![Проход по балансам строит индекс; проход по профилям берёт из него значения по ключу.](assets/indexed-join.svg)

In [ ]:
def make_balance_report(profiles, balances):
    balance_by_client = {}
    for row in balances:
        client_id = row["client_id"]
        if client_id in balance_by_client:
            raise ValueError(f"Повторный баланс клиента {client_id}")
        balance_by_client[client_id] = row["balance"]

    report = []
    for profile in profiles:
        client_id = profile["client_id"]
        if client_id not in balance_by_client:
            raise ValueError(f"Нет баланса клиента {client_id}")
        enriched = profile.copy()
        enriched["balance"] = balance_by_client[client_id]
        report.append(enriched)
    return report

original_profiles = [profile.copy() for profile in profiles]
original_balances = [row.copy() for row in balances]
report = make_balance_report(profiles, balances)
print(report)
assert report == report_by_scan
assert report[0] is not profiles[0]
assert profiles == original_profiles
assert balances == original_balances

In [ ]:
for incorrect_balances in [
    [balances[0]],  # Для Анны нет баланса.
    balances + [{"client_id": "c-1", "balance": 999}],  # Повторный c-1.
]:
    try:
        make_balance_report(profiles, incorrect_balances)
    except ValueError as error:
        print(error)
    else:
        raise AssertionError("Ожидалась ошибка входных данных")

zero_balance = make_balance_report(
    [{"client_id": "c-1", "name": "Анна"}],
    [{"client_id": "c-1", "balance": 0}],
)
assert zero_balance[0]["balance"] == 0

Индекс строится за ожидаемое `O(m)`, проход по профилям занимает ожидаемое `O(n)`. Итого `O(n + m)` времени и `O(m)` дополнительной памяти без учёта результата. Новые словари добавляются в порядке профилей. `.copy()` создаёт новый внешний словарь; вложенные объекты, если они появятся, останутся общими.

<details>
<summary>Как сократить только сборку результата</summary>

Внутри make_balance_report, если индекс уже построен с проверкой уникальности и наличие всех нужных балансов гарантировано, второй цикл можно записать так:

```python
report = [
    profile | {"balance": balance_by_client[profile["client_id"]]}
    for profile in profiles
]
```

`|` создаёт новый словарь; поля справа заменяют совпадающие поля слева. Эта запись сокращает сборку результата. Проверка дубликатов при построении индекса всё равно нужна.

</details>

## Кэш: запоминаем ответ для полного входа

Индекс хранит уже известные данные. Кэш хранит результат вычисления, чтобы повторный запрос не запускал его снова. Возьмём чистую функцию: результат зависит только от `amount` и `tax_rate`.

Вместо дорогого расчёта здесь одно умножение. Ради него кэш обычно не нужен; короткая формула позволяет проверить механизм. Словарь живёт в памяти текущего процесса, а набор входов в примере ограничен. Для настоящего сервиса отдельно нужны правила размера кэша и удаления устаревших данных.

В список `calculated_keys` будем добавлять ключ только при настоящем вычислении; его длина покажет число вычислений.

**До запуска:** сколько вычислений потребуется для трёх вызовов: `(1000, 0.13)`, снова `(1000, 0.13)`, затем `(1000, 0.20)`?

In [ ]:
cache = {}
calculated_keys = []

def net_amount(amount, tax_rate):
    key = (amount, tax_rate)
    if key not in cache:
        calculated_keys.append(key)
        cache[key] = amount * (1 - tax_rate)
    return cache[key]

results = []
for amount, tax_rate in [(1000, 0.13), (1000, 0.13), (1000, 0.20)]:
    results.append(net_amount(amount, tax_rate))
    print(results[-1], "— вычислений:", len(calculated_keys))

assert results == [870, 870, 800]
assert len(calculated_keys) == 2
assert calculated_keys == [(1000, 0.13), (1000, 0.20)]

Повторный полный ключ вернул готовый ответ, новая ставка потребовала нового вычисления. Если оставить в ключе только сумму, две разные задачи станут неразличимыми.

**Предскажите ошибочный ответ:** что вернёт следующая функция для второй ставки?

In [ ]:
incomplete_cache = {}

def broken_net_amount(amount, tax_rate):
    if amount not in incomplete_cache:
        incomplete_cache[amount] = amount * (1 - tax_rate)
    return incomplete_cache[amount]

broken_net_amount(1000, 0.13)
wrong_result = broken_net_amount(1000, 0.20)
print("Нужно 800, получено:", wrong_result)
assert wrong_result == 870  # Ошибка продемонстрирована, а не исправлена.

## `list`, `set` или `dict`

Во всех разобранных задачах выбор начинается с вопроса, который программа задаёт данным.

| Нужная операция | Контейнер и его роль |
|---|---|
| Сохранить последовательность и повторения | `list` хранит записи по порядку |
| Проверить «уже встречали?» | `set` хранит уникальные значения |
| Найти запись, сумму или число повторов по ключу | `dict` связывает ключ с нужным значением |
| Удалить повторы, сохранив первое появление | `list` задаёт порядок, `set` ускоряет проверку |
| Присоединить данные по идентификатору | `dict` хранит заранее построенный индекс |
| Повторно использовать результат вычисления | `dict` связывает полный вход с результатом |

Быстрый поиск обычно требует подготовки и дополнительной памяти. Он не делает бесплатными обход, сортировку или проверку исходных данных.

### Проверьте выбор на новой задаче

Сервис принимает платежи. У каждого платежа уникальный `payment_id`, но один клиент может платить много раз. Нужно показывать журнал в порядке поступления, находить платёж по идентификатору и знать число платежей каждого клиента.

Какие структуры вы заведёте? Для каждого словаря назовите ключ и значение. Что обновится при новом платеже? Почему одного множества клиентов недостаточно?

<details>
<summary>Сравнить рассуждение</summary>

Список хранит журнал по порядку. Словарь `payment_id → запись` даёт доступ к конкретному платежу. Словарь `client_id → количество` хранит число платежей, которое множество потеряло бы.

При поступлении нового платежа проверяем уникальность его идентификатора, добавляем запись в журнал и индекс, увеличиваем счётчик клиента. Если журнал и индекс ссылаются на одну запись, изменение её полей через индекс будет видно и в журнале: здесь общая ссылка может быть намеренной.

</details>

## Неожиданно, но по правилам

Встроенный `dict` хранит порядок вставки. Из этого нельзя заключить, что номера корзин возрастают в том же порядке: гарантия относится к поведению словаря, а наша учебная модель её не реализует.

**До запуска:** переместится ли `done` в конец при замене значения? А после удаления и новой вставки?

In [ ]:
status = {"new": 3, "done": 8, "error": 1}
status["done"] = 9
print("После замены:", list(status))
assert list(status) == ["new", "done", "error"]
del status["done"]
status["done"] = 10
print("После удаления и вставки:", list(status))
assert list(status) == ["new", "error", "done"]

## Что таблица не обещает

Быстрый доступ по ключу не даёт сортировку значений или поиск десяти самых крупных платежей. Множество отвечает на вопрос о наличии, но не хранит суммы повторных операций и порядок первого появления.

Учебные цепочки показывают размещение и проверку ключей. У встроенных контейнеров Python есть собственные внутренние решения и гарантии; переносить на них все детали нашей модели нельзя.

## Итоги

- Сначала выбираем корзину по хешу, затем проверяем идентичность или равенство ключей.
- Коллизия сохраняет разные записи в одной цепочке; замена меняет значение существующей пары.
- Новая ёмкость требует нового размещения всех пар. Простое добавление корзин ломает поиск.
- Средняя `O(1)` требует хорошего распределения и контроля заполнения. Худший случай остаётся отдельным вопросом.
- В прикладной задаче сначала определяем ключ, хранимое значение и действие при повторе. Из этого получаются накопление, дедупликация, индекс и кэш.

Дальше — дополнительные детали о необычных ключах и хешах строк. Основной механизм таблицы уже разобран.

## Дополнительно: идентичность ключей и хеши строк

Этот блок можно прочитать самостоятельно после основной части. Он объясняет две границы привычной интуиции: ключ не всегда равен самому себе, а значение `hash()` не является постоянным идентификатором.

### Почему одного `==` недостаточно

`nan` — специальное значение типа `float`, которое не равно даже самому себе. Но словарь должен находить уже вставленный объект ключа. Проверка `is` позволяет повторно обратиться к этому объекту, заменить его значение и удалить пару. Другой объект `float("nan")` будет отдельным ключом: он не идентичен первому и не равен ему.

![Повторная вставка того же объекта NaN обновляет значение; другой объект NaN создаёт отдельную пару.](assets/identity-keys.svg)

In [ ]:
nan_key = float("nan")
assert nan_key != nan_key

nan_table = make_table(1)  # Все ключи попадут в одну корзину.
assert table_put(nan_table, nan_key, "first") is True
assert table_get(nan_table, nan_key) == "first"
assert table_put(nan_table, nan_key, "updated") is False
assert len(nan_table[0]) == 1
assert table_get(nan_table, nan_key) == "updated"

other_nan = float("nan")
assert table_put(nan_table, other_nan, "second") is True
assert len(nan_table[0]) == 2
assert table_delete(nan_table, nan_key) is True
assert table_delete(nan_table, nan_key) is False
assert table_get(nan_table, other_nan) == "second"

### `hash("USD")` не обязан переживать перезапуск Python

Внутри одного процесса хеш остаётся согласованным, иначе поиск не работал бы. Но хеши `str` и `bytes` рандомизируются между процессами. Поэтому результат `hash()` нельзя записывать в файл как постоянный идентификатор.

In [ ]:
first = hash("USD")
second = hash("USD")
assert first == second
print("Хеш в текущем процессе:", first)

## Источники

- [Встроенные типы Python: `dict`](https://docs.python.org/3.14/library/stdtypes.html#mapping-types-dict) — операции отображений, требования к ключам и порядок вставки.
- [Проверка принадлежности в Python](https://docs.python.org/3.14/reference/expressions.html#membership-test-operations) — совпадение по идентичности или равенству, в том числе для ключей `dict`.
- [Модель данных Python: `object.__hash__`](https://docs.python.org/3.14/reference/datamodel.html#object.__hash__) — контракт равенства и хеша, а также рандомизация хешей строк и байтов.
- [Design and History FAQ: dictionaries](https://docs.python.org/3.14/faq/design.html#how-are-dictionaries-implemented-in-cpython) — обзор связи хеша, поиска и коллизий в CPython.